## FABlib API References Examples

- [fablib.show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config)
- [fablib.list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites)
- [fablib.list_hosts](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_hosts)
- [fablib.new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice)
- [slice.add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node)
- [slice.submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit)
- [slice.get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes)
- [slice.list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodesß)
- [slice.show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show)
- [node.execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute)
- [slice.delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete)

## Main purpose

Use this file the very first time to configure the whole Kubernetes cluster.

In [1]:
import datetime
import json
import asyncio
from configuration import SLICE_NAME
from utils import upload_and_execute_file, override_configuration_files, upload_file, execute_file

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

User: apipilikas@gmail.com bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,49f65ad7-d8a2-4ab9-8ca0-ba777a2e0ea2
Bastion Host,bastion.fabric-testbed.net
Bastion Username,apipilikas_0000444352
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


In [2]:
%%time
image = "default_ubuntu_24"

# Please adhere to the following regex for naming: /[a-z][a-z0-9]+/
# note: see above, renamed the agent names to only have hyphens, not underscores 

node_configurations = [
    {
        "type": "control",
        "cores": 2,
        "ram": 8,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    },
    {
        "type": "dynamos",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "server",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "aggregator",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "authority",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clientone",
        "cores": 4,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clienttwo",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    },
    {
        "type": "agent",
        "name": "clientthree",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    },
    {
        "type": "thirdparty",
        "name": "surf",
        "cores": 8,
        "ram": 16,
        "disk": 100,
        "site": "AMST",
        "host": "amst-w2.fabric-testbed.net",
    }
]

sites = list(set([configuration["site"] for configuration in node_configurations]))
agents = [configuration["name"] for configuration in node_configurations if configuration["type"] == "agent"]
thirdparties = [configuration["name"] for configuration in node_configurations if configuration["type"] == "thirdparty"]

def create_node(slice, configuration):
    if (configuration["type"] == "control"): 
        configuration["name"] = "control"

    if (configuration["type"] == "dynamos"): 
        configuration["name"] = "dynamos"
    
    return slice.add_node(name=configuration["name"], 
                          site=configuration["site"], 
                          host=configuration["host"], 
                          cores=configuration["cores"], 
                          ram=configuration["ram"], 
                          disk=configuration["disk"], 
                          validate=True, 
                          raise_exception=True, 
                          image=image)
    

CPU times: user 15 μs, sys: 2 μs, total: 17 μs
Wall time: 21.7 μs


In [3]:
%%time
# Create a slice
slice = fablib.new_slice(name=SLICE_NAME)

# Add Nodes with the specific variables
# Also validate the node can be created and raise an exception in case of failure
print('Adding nodes...')
nodes = [create_node(slice, configuration) for configuration in node_configurations]
nodes_per_site = [
    (site, [node for node in nodes if node.get_site() == site])
    for site in sites
]

print('Adding network interfaces...')
interfaces_per_site = [
    (site, [node.add_component(model='NIC_Basic', name='NIC').get_interfaces()[0] for node in nodes])
    for (site, nodes) in nodes_per_site
]

print('Adding network...')
networks = [
    slice.add_l3network(name=f'Network-{site}', interfaces=interfaces, type="IPv4")
    for (site, interfaces) in interfaces_per_site
]

print(networks, [n.get_gateway() for n in networks], [n.get_subnet() for n in networks])

# Calculate the lease end time for 2 weeks from now with timezone information
lease_end_time = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(weeks=2)

# Submit the slice, using an end date 2 weeks from now (the current maximum lease time) 
# to make sure that the slice can be used for a longer period of time. Progress shows an indicator of the current progression.
# Wait until the state is finished and use an interval (it may take some time before the slice and nodes are created)
print('Creating slice...')
slice.submit(wait=True, wait_timeout=3600, wait_interval=20, progress=True, wait_jupyter='text', lease_end_time=lease_end_time);


Retry: 11, Time: 355 sec


ID,a1bca089-2b0c-45bb-ae55-4fad3c5b954d
Name,S-Drive-on-FABRIC
Lease Expiration (UTC),2026-06-20 18:07:41 +0000
Lease Start (UTC),2026-06-06 18:07:41 +0000
Project ID,49f65ad7-d8a2-4ab9-8ca0-ba777a2e0ea2
State,StableOK
Email,apipilikas@gmail.com
UserId,978ed605-840c-4c61-8296-68e45dbe1c32


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
1b520587-ea7b-406e-b2dd-27e73971d7f8,aggregator,4,16,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fef7:ced3,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fef7:ced3,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
1f321dc6-85c1-4fe9-845a-25788e0555f3,authority,4,16,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe4a:4d9a,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe4a:4d9a,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
0dbb37cc-584b-4907-bbc7-e85410e85b18,clientone,4,16,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe54:989,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe54:989,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
5bcaf744-6b01-4cdb-b3be-54fcf790f4fd,clientthree,8,16,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe1c:e52c,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe1c:e52c,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
250b047b-07ca-460a-9a8f-dd04ddc17af2,clienttwo,8,16,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe7c:4f5a,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe7c:4f5a,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
fb29f19f-0c62-4d74-81fc-d8ed6186aa79,control,2,8,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe67:fd10,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
1dd77ce4-1f60-48e5-8b8c-3cca66d80162,dynamos,4,16,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe98:1cce,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe98:1cce,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
cff31b63-7f94-4886-820b-df77151104e3,server,4,16,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe50:837d,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe50:837d,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
1f52d73b-6f0a-4140-9d55-94c9344c08aa,surf,8,16,100,default_ubuntu_24,qcow2,amst-w2.fabric-testbed.net,AMST,ubuntu,2001:610:2d0:fabc:f816:3eff:fe9e:98e1,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe9e:98e1,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
e9b66eb6-9b60-47f1-b491-592af191a0ff,Network-AMST,L3,FABNetv4,AMST,10.145.1.0/24,10.145.1.1,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
control-NIC-p1,p1,control,Network-AMST,100,config,,02:A4:6F:65:1B:1F,enp7s0,enp7s0,fe80::a4:6fff:fe65:1b1f,4,HundredGigE0/0/0/7
dynamos-NIC-p1,p1,dynamos,Network-AMST,100,config,,02:DC:6F:4E:57:AF,enp7s0,enp7s0,fe80::dc:6fff:fe4e:57af,4,HundredGigE0/0/0/7
server-NIC-p1,p1,server,Network-AMST,100,config,,06:94:D5:68:43:2B,enp7s0,enp7s0,fe80::494:d5ff:fe68:432b,4,HundredGigE0/0/0/7
aggregator-NIC-p1,p1,aggregator,Network-AMST,100,config,,0E:EA:F8:31:B9:58,enp7s0,enp7s0,fe80::cea:f8ff:fe31:b958,4,HundredGigE0/0/0/7
authority-NIC-p1,p1,authority,Network-AMST,100,config,,16:09:E0:EC:F7:F2,enp7s0,enp7s0,fe80::1409:e0ff:feec:f7f2,4,HundredGigE0/0/0/7
clientone-NIC-p1,p1,clientone,Network-AMST,100,config,,12:6D:67:78:0D:28,enp7s0,enp7s0,fe80::106d:67ff:fe78:d28,4,HundredGigE0/0/0/7
clienttwo-NIC-p1,p1,clienttwo,Network-AMST,100,config,,1A:1F:7F:E1:3B:30,enp7s0,enp7s0,fe80::181f:7fff:fee1:3b30,4,HundredGigE0/0/0/7
clientthree-NIC-p1,p1,clientthree,Network-AMST,100,config,,1E:17:F8:12:AD:0A,enp7s0,enp7s0,fe80::1c17:f8ff:fe12:ad0a,4,HundredGigE0/0/0/7
surf-NIC-p1,p1,surf,Network-AMST,100,config,,1E:7F:6D:59:7D:8A,enp7s0,enp7s0,fe80::1c7f:6dff:fe59:7d8a,4,HundredGigE0/0/0/7



Time to print interfaces 376 seconds
CPU times: user 1min 45s, sys: 414 ms, total: 1min 45s
Wall time: 7min 8s


In [4]:
%%time
slice = fablib.get_slice(name=SLICE_NAME);
nodes = slice.get_nodes();

nodes_and_network_per_site = [
    (site, [node for node in nodes if node.get_site() == site], slice.get_network(name=f"Network-{site}"))
    for site in sites
]
networks = [network for (_, _, network) in nodes_and_network_per_site]

nodes_network_ips_per_site = [
    (site, nodes, network, network.get_available_ips(len(nodes)))
    for (site, nodes, network) in nodes_and_network_per_site
]

CPU times: user 1.91 s, sys: 1.96 ms, total: 1.91 s
Wall time: 2.39 s


In [5]:
%%time
def assign_ip(site, network, available_ips, node):
    interface = node.get_interface(network_name=f"Network-{site}")
    address = available_ips.pop(0)
    network_gateway = network.get_gateway()
    network_subnet = network.get_subnet()

    network.allocate_ip(address)
    interface.ip_addr_add(addr=address, subnet=network_subnet)
    node.ip_route_add(subnet=network_subnet, gateway=network_gateway)

    # For the multisite IPv4 connection
    for network in networks:
        node.ip_route_add(subnet=network.get_subnet(), gateway=network_gateway)

    return address

ips = [assign_ip(site, network, ips, node) for (site, nodes, network, ips) in nodes_network_ips_per_site for node in nodes];

CPU times: user 4.11 s, sys: 117 ms, total: 4.23 s
Wall time: 44.4 s


In [6]:
%%time
slice = fablib.get_slice(name=SLICE_NAME);
nodes = slice.get_nodes();

for node in nodes:
    print(node.get_name())
    ssh_command = node.get_ssh_command().replace(
        "-i /home/fabric/work/fabric_config/slice_key", "-i ~/.ssh/keys/FABRIC-slice_key"
    ).replace(
        "-F /home/fabric/work/fabric_config/ssh_config ", ""
    )
    
    print(ssh_command);

control
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe67:fd10
dynamos
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe98:1cce
server
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe50:837d
aggregator
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fef7:ced3
authority
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe4a:4d9a
clientone
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe54:989
clienttwo
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe7c:4f5a
clientthree
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe1c:e52c
surf
ssh -i ~/.ssh/keys/FABRIC-slice_key ubuntu@2001:610:2d0:fabc:f816:3eff:fe9e:98e1
CPU times: user 19.1 s, sys: 8.82 ms, total: 19.1 s
Wall time: 19.5 s


In [7]:
%%time
print("Uploading the node setup...")
threads = [node.upload_file_thread(local_file_path="node_scripts/node_setup.sh", remote_file_path="setup.sh")
           for node in nodes]
[thread.result() for thread in threads]


Uploading the node setup...
CPU times: user 193 ms, sys: 52.3 ms, total: 245 ms
Wall time: 2.64 s


[<SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>,
 <SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>,
 <SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>,
 <SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>,
 <SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>,
 <SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>,
 <SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>,
 <SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>,
 <SFTPAttributes: [ size=1250 uid=1000 gid=1000 mode=0o100664 atime=1780769707 mtime=1780769707 ]>]

In [8]:
%%time
print("Executing the node setup...")
threads = [node.execute_thread(f"sed -i 's/\\r$//' setup.sh && chmod +x setup.sh && ./setup.sh")
           for node in nodes]
[thread.result() for thread in threads]

Executing the node setup...
CPU times: user 8.68 s, sys: 2.04 s, total: 10.7 s
Wall time: 5min 57s


[("Get:1 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]\nHit:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease\nGet:3 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]\nGet:4 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]\nGet:5 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1761 kB]\nGet:6 http://security.ubuntu.com/ubuntu noble-security/main Translation-en [275 kB]\nGet:7 http://nova.clouds.archive.ubuntu.com/ubuntu noble/universe amd64 Packages [15.0 MB]\nGet:8 http://security.ubuntu.com/ubuntu noble-security/main amd64 Components [42.4 kB]\nGet:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 c-n-f Metadata [11.4 kB]\nGet:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1194 kB]\nGet:11 http://security.ubuntu.com/ubuntu noble-security/universe Translation-en [231 kB]\nGet:12 http://security.ubuntu.com/ubuntu noble-security/un

In [9]:
%%time
def get_ip(node):
    interface = node.get_interface(network_name=f"Network-{node.get_site()}")
    return interface.get_ip_addr()

nodes_dict= dict()

for node in nodes[:]:
    ip = get_ip(node)
    name = node.get_name()
    nodes_dict[name] = {"ip": ip, "node": node}
    print(f"{name}: {ip}")

print(nodes_dict)


control: 10.145.1.2
dynamos: 10.145.1.3
server: 10.145.1.4
aggregator: 10.145.1.5
authority: 10.145.1.6
clientone: 10.145.1.7
clienttwo: 10.145.1.8
clientthree: 10.145.1.9
surf: 10.145.1.10
{'control': {'ip': '10.145.1.2', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x79ac78437dd0>}, 'dynamos': {'ip': '10.145.1.3', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x79ac7c63ec50>}, 'server': {'ip': '10.145.1.4', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x79ac78164e10>}, 'aggregator': {'ip': '10.145.1.5', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x79ac7838d150>}, 'authority': {'ip': '10.145.1.6', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x79ac5881d2d0>}, 'clientone': {'ip': '10.145.1.7', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x79ac58c93990>}, 'clienttwo': {'ip': '10.145.1.8', 'node': <fabrictestbed_extensions.fablib.node.Node object at 0x79ac7816c410>}, 'clientthree': {'ip': '1

## Step : Installing k9s

In [10]:
%%time
# nodes_dict["control"]["node"].upload_file(local_file_path="node_scripts/install_k9s.sh", remote_file_path="k9s.sh")
# nodes_dict["control"]["node"].execute(f"chmod +x k9s.sh && ./k9s.sh");
upload_and_execute_file(nodes_dict["control"]["node"], local_file_path="node_scripts/install_k9s.sh", remote_file_path="k9s.sh")

Uploading file from local path [node_scripts/install_k9s.sh] to remote path [k9s.sh] ...
Executing file [k9s.sh] ...
--2026-06-06 18:21:28--  https://github.com/derailed/k9s/releases/download/v0.32.5/k9s_linux_amd64.deb
Resolving github.com (github.com)... 2600:2701:5000:5001::8c52:7203, 140.82.114.3
Connecting to github.com (github.com)|2600:2701:5000:5001::8c52:7203|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/167596393/7cc41638-6a22-4598-9b02-646efaaa1053?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-06-06T19%3A12%3A11Z&rscd=attachment%3B+filename%3Dk9s_linux_amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-06-06T18%3A11%3A16Z&ske=2026-06-06T19%3A12%3A11Z&sks=b&skv=2018-11-09&sig=LiO6xzOOG52rUNs5Z2p719h4L2IVeIA%2B8r%2BmfngZt%2BQ%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWI

In [11]:
# the order of the nodes is not the same as in the configuration 
# ips
# names
# names_and_ips = {n:i for n,i in names_and_ips}
# print(names_and_ips)

## Step : Creating kubespray 'inventory.ini'

In [12]:
%%time
inventory = (
    f"[kube_control_plane]\n"
    f"control ansible_host={nodes_dict['control']['ip']} ip={nodes_dict['control']['ip']} etcd_member_name=etcd1\n"
    f"\n"
    f"[etcd:children]\n"
    f"kube_control_plane\n"
    f"\n"
    f"[kube_node]\n"
    f"dynamos ansible_host={nodes_dict['dynamos']['ip']} ip={nodes_dict['dynamos']['ip']}\n"
)

for node_name in nodes_dict.keys():
    if node_name not in ["dynamos", "control"]:
        inventory += f"{node_name} ansible_host={nodes_dict[node_name]['ip']} ip={nodes_dict[node_name]['ip']}\n"

with open('kubespray/inventory.ini', 'w') as f:
    f.write(inventory)

CPU times: user 305 μs, sys: 0 ns, total: 305 μs
Wall time: 444 μs


In [13]:
print(inventory)

[kube_control_plane]
control ansible_host=10.145.1.2 ip=10.145.1.2 etcd_member_name=etcd1

[etcd:children]
kube_control_plane

[kube_node]
dynamos ansible_host=10.145.1.3 ip=10.145.1.3
server ansible_host=10.145.1.4 ip=10.145.1.4
aggregator ansible_host=10.145.1.5 ip=10.145.1.5
authority ansible_host=10.145.1.6 ip=10.145.1.6
clientone ansible_host=10.145.1.7 ip=10.145.1.7
clienttwo ansible_host=10.145.1.8 ip=10.145.1.8
clientthree ansible_host=10.145.1.9 ip=10.145.1.9
surf ansible_host=10.145.1.10 ip=10.145.1.10



In [14]:
main_node=nodes_dict['control']['node']
print(f"The main node for now on is [{main_node.get_name()}]")

The main node for now on is [control]


## Step : Setting up kubespray

In [15]:
%%time
# nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/control_kubespray_setup.sh", remote_file_path="kubespray_setup.sh");
# nodes_dict['control']['node'].execute("chmod +x kubespray_setup.sh && ./kubespray_setup.sh");
upload_and_execute_file(nodes_dict['control']['node'], local_file_path="node_scripts/control_kubespray_setup.sh", remote_file_path="kubespray_setup.sh")

nodes_dict['control']['node'].upload_file(local_file_path="kubespray/inventory.ini", remote_file_path="kubespray/inventory/dynamos/inventory.ini");
nodes_dict['control']['node'].upload_file(local_file_path="kubespray/ansible.cfg", remote_file_path="kubespray/ansible.cfg");
# nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/dot_kube.sh", remote_file_path="dot_kube.sh");
# nodes_dict['control']['node'].execute("chmod +x ./dot_kube.sh");
# upload_and_execute_file(nodes_dict['control']['node'], local_file_path="node_scripts/dot_kube.sh", remote_file_path="dot_kube.sh")
nodes_dict['control']['node'].upload_file(local_file_path="/home/fabric/work/fabric_config/slice_key", remote_file_path="/home/ubuntu/.ssh/slice_key");
nodes_dict['control']['node'].execute("chmod 600 /home/ubuntu/.ssh/slice_key");

Uploading file from local path [node_scripts/control_kubespray_setup.sh] to remote path [kubespray_setup.sh] ...
Executing file [kubespray_setup.sh] ...
Cloning into 'kubespray'...
branch 'release-2.27' set up to track 'origin/release-2.27'.
Switched to a new branch 'release-2.27'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.6/219.6 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 14.0 MB/s eta 0:00:00
ans

## Step : Installing kubespray (>!\< Takes time >!\<)

In [ ]:
%%time
# 27 mins , 20 mins , 10 min at CIEN
# nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/start_kubespray.sh", remote_file_path="start.sh");
# nodes_dict['control']['node'].execute(f"chmod +x start.sh && ./start.sh");
upload_and_execute_file(nodes_dict['control']['node'], local_file_path="node_scripts/start_kubespray.sh", remote_file_path="start.sh")

Uploading file from local path [node_scripts/start_kubespray.sh] to remote path [start.sh] ...
Executing file [start.sh] ...
[WARNING]: While constructing a mapping from
/home/ubuntu/kubespray/roles/bootstrap-os/tasks/main.yml, line 29, column 7,
found a duplicate dict key (paths). Using last defined value only.
[WARNING]: Skipping callback plugin 'ara_default', unable to load
Using /home/ubuntu/kubespray/ansible.cfg as config file

PLAY [Check Ansible version] ***************************************************
Saturday 06 June 2026  18:22:29 +0000 (0:00:00.008)       0:00:00.008 ********* 

TASK [Check 2.16.4 <= Ansible version < 2.17.0] ********************************
ok: [dynamos] => {
    "changed": false,
    "msg": "All assertions passed"
}
Saturday 06 June 2026  18:22:29 +0000 (0:00:00.020)       0:00:00.028 ********* 

TASK [Check that python netaddr is installed] **********************************
ok: [dynamos] => {
    "changed": false,
    "msg": "All assertions passed"
}


In [16]:
# This is for resetting the kubespray cluster. 
# Use this if you are troubleshooting your Kubernetes cluster
# and you want to redeploy fresh.

# nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/reset_kubespray.sh", remote_file_path="reset.sh");
# nodes_dict['control']['node'].execute(f"chmod +x reset.sh && ./reset.sh");

## Step : Cloning repository

In [17]:
# Add the relevant etcd data to the dynamos node

# help(nodes_dict['dynamos']['node'])
# upload etcd files from filesystem instead of reading them from github
# nodes_dict['dynamos']['node'].upload_directory(local_directory_path="../configuration/etcd_launch_files", remote_directory_path="./")
# nodes_dict['dynamos']['node'].execute("ls etcd_launch_files")

upload_and_execute_file(nodes_dict['control']['node'], local_file_path="node_scripts/define_etcd_data.sh", remote_file_path="define_etcd_data.sh")

Uploading file from local path [node_scripts/define_etcd_data.sh] to remote path [define_etcd_data.sh] ...
Executing file [define_etcd_data.sh] ...
Cleaning scattered-directive-energy-monitoring folder ...
Cloning into 'scattered-directive-energy-monitoring'...


## Step : Configuring DYNAMOS

In [18]:
%%time
# Preconfigure Helm for DYNAMOS and clone DYNAMOS repo
upload_and_execute_file(nodes_dict['control']['node'], local_file_path="node_scripts/install_dynamos.sh", remote_file_path="dynamos.sh")

Uploading file from local path [node_scripts/install_dynamos.sh] to remote path [dynamos.sh] ...
Executing file [dynamos.sh] ...
Hit:1 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Get:3 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:4 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease
Get:6 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/main amd64 Components [177 kB]
Get:7 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/universe amd64 Components [386 kB]
Get:8 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates/multiverse amd64 Components [940 B]
Hit:5 https://prod-cdn.packages.k8s.io/repositories/isv:/kubernetes:/core:/stable:/v1.30/deb  InRelease
Fetched 691 kB in 1s (511 kB/s)
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
curl is already the newest version (8.

In [19]:
# This procedure has been incorporated on the main DYNAMOS installation.
# # Configure DYNAMOS for the FABRIC nodes
# agents_string = ",".join(agents)
# thirdparties_string = ",".join(thirdparties)

# # nodes_dict['control']['node'].upload_file(local_file_path="node_scripts/configure_dynamos.sh", remote_file_path="configure_dynamos.sh"); # why upload can use the existing one
# # nodes_dict['control']['node'].execute(f"chmod +x configure_dynamos.sh && ./configure_dynamos.sh {agents_string} {thirdparties_string}");
# upload_file(nodes_dict['control']['node'], local_file_path="scripts/add_agent.sh", remote_file_path="scattered-directive-energy-monitoring/add_agent.sh");
# upload_file(nodes_dict['control']['node'], local_file_path="scripts/add_thirdparty.sh", remote_file_path="scattered-directive-energy-monitoring/add_thirdparty.sh");
# upload_and_execute_file(nodes_dict['control']['node'], local_file_path="node_scripts/configure_dynamos.sh", remote_file_path="configure_dynamos.sh", script_args=f"{agents_string} {thirdparties_string}")

## Step : Overriding files

In [20]:
# Optionally override the installation scripts 
override_configuration_files(main_node)
main_node.execute("find /home/ubuntu/scattered-directive-energy-monitoring -type f -name '*.sh' -exec sed -i 's/\\r$//' {} + -exec chmod +x {} +")

Uploading file from local path [dynamos.conf] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/dynamos.conf] ...
Uploading file from local path [overriden_files/temp-pod.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/configuration/temp-pod.yaml] ...
Uploading file from local path [overriden_files/core-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/core/values.yaml] ...
Uploading file from local path [overriden_files/orchestrator-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/orchestrator/values.yaml] ...
Uploading file from local path [overriden_files/namespaces-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/namespaces/values.yaml] ...
Uploading file from local path [overriden_files/api-gateway-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/api-gateway/value

('', '')

In [21]:
# For some reason, even though node_setup installed docker, seems like it did not. It has to be re-executed.
nodes_dict['control']['node'].execute("sudo apt-get update && sudo apt-get install -y docker.io apt-transport-https curl python3 python3-venv python3-pip ca-certificates gpg")
nodes_dict['control']['node'].execute("sudo systemctl restart docker")
nodes_dict['control']['node'].execute("sudo chmod 666 /var/run/docker.sock")

Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease
Hit:4 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:6 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:2 https://prod-cdn.packages.k8s.io/repositories/isv:/kubernetes:/core:/stable:/v1.30/deb  InRelease
Hit:5 https://packages.buildkite.com/helm-linux/helm-debian/any any InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
apt-transport-https is already the newest version (2.8.3).
curl is already the newest version (8.5.0-2ubuntu10.9).
python3 is already the newest version (3.12.3-0ubuntu2.1).
python3-venv is already the newest version (3.12.3-0ubuntu2.1).
python3-pip is already the newest version (24.0+dfsg-1ubuntu1.3).
ca-certificates is already the newest version (20240203).
gpg is already the newest version (2.4.4-2ubuntu17.4).
The follow

('', '')

## Step : Installing DYNAMOS

In [22]:
%%time
# install DYNAMOS
nodes[0].execute("cd /home/ubuntu/scattered-directive-energy-monitoring/configuration && sed -i 's/\\r$//' *.sh && chmod +x *.sh")
nodes_dict['control']['node'].execute(f"sed -i 's/\\r$//' ~/scattered-directive-energy-monitoring/configuration/dynamos-configuration.sh && ~/scattered-directive-energy-monitoring/configuration/dynamos-configuration.sh fabric")


=============== Started setting up DYNAMOS (fabric) ===============

Setting up paths...

Agents discovered: aggregator,authority,server,clientone,clienttwo,clientthree

Generating agents and third parties charts...

Adding agents...
- agent 'aggregator'
---

apiVersion: v1
kind: ServiceAccount
metadata:
  name: job-creator-aggregator
  namespace: aggregator
---
apiVersion: rbac.authorization.k8s.io/v1
kind: RoleBinding
metadata:
  name: job-creator-aggregator
  namespace: aggregator
roleRef:
  apiGroup: rbac.authorization.k8s.io
  kind: ClusterRole
  name: job-creator
subjects:
- kind: ServiceAccount
  name: job-creator-aggregator
  namespace: aggregator
---

apiVersion: v1
kind: Namespace
metadata:
  name: aggregator
  annotations:
    "helm.sh/resource-policy": keep
    "app.kubernetes.io/managed-by": "Helm"
    "config.linkerd.io/trace-collector": collector.linkerd-jaeger:55678 # or 14268?

---

apiVersion: v1
kind: Secret
metadata:
  name: rabbit
  namespace: aggregator
type: Opa

('\n=============== Started setting up DYNAMOS (fabric) ===============\n\nSetting up paths...\n\nAgents discovered: aggregator,authority,server,clientone,clienttwo,clientthree\n\nGenerating agents and third parties charts...\n\nAdding agents...\n- agent \'aggregator\'\n---\n\napiVersion: v1\nkind: ServiceAccount\nmetadata:\n  name: job-creator-aggregator\n  namespace: aggregator\n---\napiVersion: rbac.authorization.k8s.io/v1\nkind: RoleBinding\nmetadata:\n  name: job-creator-aggregator\n  namespace: aggregator\nroleRef:\n  apiGroup: rbac.authorization.k8s.io\n  kind: ClusterRole\n  name: job-creator\nsubjects:\n- kind: ServiceAccount\n  name: job-creator-aggregator\n  namespace: aggregator\n---\n\napiVersion: v1\nkind: Namespace\nmetadata:\n  name: aggregator\n  annotations:\n    "helm.sh/resource-policy": keep\n    "app.kubernetes.io/managed-by": "Helm"\n    "config.linkerd.io/trace-collector": collector.linkerd-jaeger:55678 # or 14268?\n\n---\n\napiVersion: v1\nkind: Secret\nmetadat

In [16]:
# Optional to clean up: uninstall DYNAMOS
cleanup_command = """
helm uninstall surf -n default --ignore-not-found

kubectl delete serviceaccount job-creator-surf -n default --ignore-not-found
kubectl delete rolebinding job-creator-surf -n default --ignore-not-found
kubectl delete clusterrolebinding job-creator-surf --ignore-not-found
"""
nodes_dict['control']['node'].execute(cleanup_command)
upload_and_execute_file(nodes_dict['control']['node'], "scripts/uninstall-dynamos.sh", "uninstall-dynamos.sh")

# command = "helm uninstall agents api-gateway core orchestrator namespaces prometheus thirdparties"
# nodes_dict['control']['node'].execute(command)

release "surf" uninstalled
Uploading file from local path [scripts/uninstall-dynamos.sh] to remote path [uninstall-dynamos.sh] ...
Executing file [uninstall-dynamos.sh] ...
Uninstalling DYNAMOS namespaces...
./uninstall-dynamos.sh: line 4: /home/ubuntu/../dynamos.conf: No such file or directory
release "nginx" uninstalled
These resources were kept due to the resource policy:
[Namespace] core
[Namespace] orchestrator
[Namespace] clientthree
[Namespace] uva
[Namespace] vu
[Namespace] surf
[Namespace] ingress
[Namespace] api-gateway
[Namespace] server
[Namespace] clientone
[Namespace] clienttwo

release "namespaces" uninstalled
release "core" uninstalled
release "orchestrator" uninstalled
release "agents" uninstalled
release "thirdparties" uninstalled
release "api-gateway" uninstalled
release "surf" uninstalled
Uninstalling monitoring namespaces...
release "prometheus" uninstalled
release "grafana" uninstalled
Error: uninstall: Release not loaded: kepler: release: not found
Uninstalling n

In [ ]:
# Optional delete etcd PVCs 
# nodes_dict['control']['node'].execute("kubectl get pvc --all-namespaces")

# nodes_dict['control']['node'].execute("kubectl delete pvc etcd-data-etcd-0 -n core")
# nodes_dict['control']['node'].execute("kubectl delete pvc etcd-data-etcd-1 -n core")
# nodes_dict['control']['node'].execute("kubectl delete pvc etcd-data-etcd-2 -n core")


# nodes_dict['control']['node'].execute("kubectl get pvc --all-namespaces")